# Aula 03 — Pré-processamento e Índice Invertido (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel)*

In [1]:
# 1. Setup oficial do motor03 e inspecao do estado herdado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor03.R")
docs <- docs_aula()
cfg  <- cfg_aula()
ix   <- montar(docs, cfg)
estado()

[1] "MOTOR_VERSAO: motor03 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 0 | elementos: |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao ordena..., d2 o modelo de espaco vetorial repre..., ... |
| `ix` | list | 8 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn |


## Módulo 1 — Por que pré-processar: o texto sujo
Demonstração do impacto de pontuações, maiúsculas e caracteres especiais na tokenização e busca ingênua.

In [2]:
# Módulo 1: O corpus bruto com pontuacao e caixas variadas
brutos <- c(
  d1 = "Recuperacao de Informacao: ORDENA documentos, por relevancia!",
  d2 = "O modelo de espaco-vetorial representa documentos (como vetores).",
  d3 = "BM25 e um modelo probabilistico de ranqueamento de texto.",
  d4 = "Aprendizado estatistico fundamenta a recuperacao moderna.",
  d5 = "O indice invertido acelera a busca em muitos documentos.",
  d6 = "Embeddings capturam a semantica de palavras e documentos.",
  d7 = "A avaliacao mede a relevancia dos resultados da busca.",
  d8 = "Ciencia de dados combina estatistica e programacao."
)

cat("Tokenizacao crua de d1:\n")
print(tokenizar(brutos[["d1"]]))

cat("\nTokenizacao crua de d2:\n")
print(tokenizar(brutos[["d2"]]))

cat("\nTamanho do vocabulario sujo vs limpo:\n")
cat("Sujo:", length(unique(unlist(lapply(brutos, tokenizar)))), "| Limpo:", length(ix$vocab), "\n")

# TDM com tokens sujos
bruto_tf <- matriz_tf(lapply(brutos, tokenizar))
cat("\nBusca booleana por 'documentos' (sujo vs limpo):\n")
cat("Sujo :", busca_booleana("documentos", bruto_tf), "\n")
cat("Limpo:", busca_booleana("documentos", ix$tf), "\n")

cat("\nBusca booleana por 'busca' (Checkpoint 1):\n")
cat("Sujo :", busca_booleana("busca", bruto_tf), "\n")
cat("Limpo:", busca_booleana("busca", ix$tf), "\n")

Tokenizacao crua de d1:
[1] "recuperacao" "de"          "informacao:" "ordena"      "documentos,"
[6] "por"         "relevancia!"

Tokenizacao crua de d2:
[1] "o"               "modelo"          "de"              "espaco-vetorial"
[5] "representa"      "documentos"      "(como"           "vetores)."      

Tamanho do vocabulario sujo vs limpo:
Sujo: 48 | Limpo: 45 

Busca booleana por 'documentos' (sujo vs limpo):
Sujo : d2 
Limpo: d1 d2 d5 d6 

Busca booleana por 'busca' (Checkpoint 1):
Sujo : d5 
Limpo: d5 d7 


## Módulo 2 — `limpar`: as duas regex
Construção do pipeline determinístico: minúsculas, remoção de caracteres não-alfanuméricos, colapso de espaços e `trimws`.

In [3]:
# Módulo 2: Funcao limpar com expressao regular e vetorizacao
limpar <- function(x) {
  x <- tolower(x)
  x <- gsub("[^a-z0-9 ]", " ", x)
  x <- gsub("\\s+", " ", x)
  trimws(x)
}

cat("d1 limpo:\n")
print(limpar(brutos[["d1"]]))

# Aplicando a todos os documentos
limpos <- limpar(brutos)
cat("\nTres primeiros documentos limpos:\n")
print(head(limpos, 3))

cat("\nTodos os documentos limpos sao identicos ao docs canonico?\n")
print(all(limpos == docs))

# Casos de teste
cat("\nCasos de teste e acentos:\n")
cat("Porto de Santos: ", limpar("Porto de Santos: 1º lugar!"), "\n")
cat("Espacos pontas : ", limpar("  Ilha   Porchat  "), "\n")
cat("Com acento cru : ", limpar("Olá, Cubatão"), "\n")
cat("Com translit   : ", limpar(iconv("Olá, Cubatão", from = "UTF-8", to = "ASCII//TRANSLIT")), "\n")
cat("Checkpoint 2   : ", limpar("R$ 3,50 (meia-entrada)"), "\n")

d1 limpo:
[1] "recuperacao de informacao ordena documentos por relevancia"

Tres primeiros documentos limpos:
                                                              d1 
    "recuperacao de informacao ordena documentos por relevancia" 
                                                              d2 
"o modelo de espaco vetorial representa documentos como vetores" 
                                                              d3 
      "bm25 e um modelo probabilistico de ranqueamento de texto" 

Todos os documentos limpos sao identicos ao docs canonico?
[1] TRUE

Casos de teste e acentos:
Porto de Santos:  porto de santos 1 lugar 
Espacos pontas :  ilha porchat 
Com acento cru :  ol cubat o 
Com translit   :  ola cubatao 
Checkpoint 2   :  r 3 50 meia entrada 


## Módulo 3 — Stopwords
Demonstração matemática de como palavras vazias desviam a ordenação vetorial e implementação do filtro `sem_stop`.

In [4]:
# Módulo 3: Impacto de palavras funcionais e remocao de stopwords
cat("Ranking com 'de':\n")
print(round(ranking_cosseno("modelo de recuperacao", ix), 3))

cat("\nRanking sem 'de':\n")
print(round(ranking_cosseno("modelo recuperacao", ix), 3))

# Pipeline sem_stop
stopwords <- c("de", "o", "a", "e", "um", "por", "como", "que", "da", "do")
tok <- function(x) unlist(strsplit(limpar(x), " "))
sem_stop <- function(x) {
  t <- tok(x)
  t[!t %in% stopwords]
}

cat("\nsem_stop(d2):\n")
print(sem_stop(brutos[["d2"]]))

cat("\nsem_stop(d8):\n")
print(sem_stop(brutos[["d8"]]))

cat("\nsem_stop(d5) (sobrevivencia de 'em'):\n")
print(sem_stop(brutos[["d5"]]))

cat("\nStopwords presentes no corpus (8 das 10):\n")
print(stopwords[stopwords %in% ix$vocab])

cat("\nCheckpoint 3 (sem_stop em d7):\n")
print(sem_stop(brutos[["d7"]]))

Ranking com 'de':
   d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000 

Ranking sem 'de':
   d1    d4    d3    d2    d5    d6    d7    d8 
0.234 0.221 0.195 0.192 0.000 0.000 0.000 0.000 

sem_stop(d2):
[1] "modelo"     "espaco"     "vetorial"   "representa" "documentos"
[6] "vetores"   

sem_stop(d8):
[1] "ciencia"     "dados"       "combina"     "estatistica" "programacao"

sem_stop(d5) (sobrevivencia de 'em'):
[1] "indice"     "invertido"  "acelera"    "busca"      "em"        
[6] "muitos"     "documentos"

Stopwords presentes no corpus (8 das 10):
[1] "de"  "o"   "a"   "e"   "um"  "por" "como" "da" 

Checkpoint 3 (sem_stop em d7):
[1] "avaliacao"  "mede"       "relevancia" "dos"        "resultados"
[6] "busca"     


## Módulo 4 — Stemming: brinquedo e Snowball
Redução morfológica a radicais. Comparação entre regra ingênua e o algoritmo canônico Snowball para a língua portuguesa.

In [5]:
# Módulo 4: Reducao morfologica por radical (stemming)
stem <- function(t) sub("(cao|mento|dade|ais|s)$", "", t)

cat("Stemmer de brinquedo:\n")
print(sapply(c("documentos", "ranqueamento", "relevancia", "modelos"), stem))

cat("\nCasos limites do brinquedo:\n")
print(sapply(c("modelo", "modelos", "mais", "recuperacao"), stem))

# SnowballC
if (!requireNamespace("SnowballC", quietly = TRUE)) {
  install.packages("SnowballC", quiet = TRUE)
}
library(SnowballC)

cat("\nSnowball em portugues com acento:\n")
print(wordStem(c("documentos", "documento", "documentação"), language = "portuguese"))

cat("\nSnowball sem acento:\n")
print(wordStem("documentacao", language = "portuguese"))

cat("\nCheckpoint 4: 'cidades' e 'cidade':\n")
cat("Brinquedo: ", stem("cidades"), "vs", stem("cidade"), "\n")
cat("Snowball : ", wordStem(c("cidades", "cidade"), language = "portuguese"), "\n")

Stemmer de brinquedo:
  documentos ranqueamento   relevancia      modelos 
 "documento"    "ranquea" "relevancia"     "modelo" 

Casos limites do brinquedo:
       modelo       modelos          mais   recuperacao 
     "modelo"      "modelo"           "m"    "recupera" 

Snowball em portugues com acento:
[1] "document" "document" "document"

Snowball sem acento:
[1] "documentaca"

Checkpoint 4: 'cidades' e 'cidade':
Brinquedo:  cidade vs ci 
Snowball :  cidad cidad 


## Módulo 5 — Tokenização subword: o problema e as famílias
Análise do problema de OOV (Out-of-Vocabulary) e visão comparativa entre BPE, WordPiece e SentencePiece.

In [6]:
# Módulo 5: Demonstracao do problema de OOV no modelo de espaco vetorial
cat("Consulta OOV 'santos' no motor vetorial (retorna todos 0):\n")
print(ranking_cosseno("santos", ix))

cat("\n'documento' existe no vocabulario?\n")
print("documento" %in% ix$vocab)

cat("'documentos' existe no vocabulario?\n")
print("documentos" %in% ix$vocab)

Consulta OOV 'santos' no motor vetorial (retorna todos 0):
d1 d2 d3 d4 d5 d6 d7 d8 
 0  0  0  0  0  0  0  0 

'documento' existe no vocabulario?
[1] FALSE

'documentos' existe no vocabulario?
[1] TRUE


## Módulo 6 — BPE (Byte Pair Encoding) em três passos
Construção didática passo a passo do algoritmo de fusão estatística de subwords por contagem de pares vizinhos.

In [7]:
# Módulo 6: Implementacao didatica de BPE
freq <- c(praias = 5, ilhas = 4, ruas = 3, praia = 2)

# Passo 1: Separacao em caracteres
simbolos <- lapply(names(freq), function(p) strsplit(p, "")[[1]])
names(simbolos) <- names(freq)
cat("Passo 1: Simbolos iniciais de 'praias':\n")
print(simbolos[["praias"]])

# Passo 2: Contar pares ponderados
contar_pares <- function(simb, freq) {
  pares <- integer(0)
  for (p in names(simb)) {
    s <- simb[[p]]
    for (i in seq_len(length(s) - 1)) {
      k <- paste0(s[i], s[i + 1])
      pares[k] <- ifelse(is.na(pares[k]), 0, pares[k]) + freq[[p]]
    }
  }
  sort(pares, decreasing = TRUE)
}

cat("\nPasso 2: Placar de pares ponderados (Top 4):\n")
print(head(contar_pares(simbolos, freq), 4))

# Passo 3: Fusao do par vencedor ('a s' -> 'as')
palavras <- c(praias = "p r a i a s", ilhas = "i l h a s", ruas = "r u a s", praia = "p r a i a")
fundidas <- gsub("a s", "as", palavras)
cat("\nPasso 3: Palavras apos fusao de 'as':\n")
print(fundidas)

# Checkpoint 6: Segunda rodada de contagem de pares
simb2 <- lapply(strsplit(fundidas, " "), function(x) x)
names(simb2) <- names(freq)
cat("\nCheckpoint 6: Placar da 2a rodada (Top 4):\n")
print(head(contar_pares(simb2, freq), 4))

Passo 1: Simbolos iniciais de 'praias':
[1] "p" "r" "a" "i" "a" "s"

Passo 2: Placar de pares ponderados (Top 4):
as pr ra ai 
12  7  7  7 

Passo 3: Palavras apos fusao de 'as':
      praias        ilhas         ruas        praia 
"p r a i as"   "i l h as"     "r u as"  "p r a i a" 

Checkpoint 6: Placar da 2a rodada (Top 4):
 pr  ra  ai ias 
  7   7   7   5 


## Módulo 7 — Por que um índice invertido
Mudança paradigmática: do modelo direto (documento $\rightarrow$ termos) para o invertido (termo $\rightarrow$ lista de documentos / postagens).

In [8]:
# Módulo 7: Inversao de perspectiva (documento->termos vs termo->documentos)
cat("Linha da TDM para 'busca' (com zeros esparsos):\n")
print(ix$tf["busca", ])

cat("\nBusca booleana correspondente:\n")
print(busca_booleana("busca", ix$tf))

cat("\nDensidade da matriz TDM:\n")
cat("Valores nao-nulos:", sum(ix$tf > 0), "de", nrow(ix$tf) * ncol(ix$tf), "posicoes.\n")

Linha da TDM para 'busca' (com zeros esparsos):
d1 d2 d3 d4 d5 d6 d7 d8 
 0  0  0  0  1  0  1  0 

Busca booleana correspondente:
[1] "d5" "d7"

Densidade da matriz TDM:
Valores nao-nulos: 62 de 360 posicoes.


## Módulo 8 — Construindo o índice em R
Implementação do índice invertido em R via listas nomeadas e laço com `unique()`.

In [9]:
# Módulo 8: Construcao da lista de postagens (indice invertido)
prep <- function(x) sem_stop(x)
postings <- list()

for (d in names(brutos)) {
  for (termo in unique(prep(brutos[[d]]))) {
    postings[[termo]] <- c(postings[[termo]], d)
  }
}

cat("Postings de 'documentos':\n")
print(postings[["documentos"]])

cat("\nPostings de 'modelo':\n")
print(postings[["modelo"]])

cat("\nPostings de 'de' (stopword removida):\n")
print(postings[["de"]])

# Comparacao sem unique e sem stopwords (p_sem)
p_sem <- list()
for (d in names(brutos)) {
  for (termo in tok(brutos[[d]])) {
    p_sem[[termo]] <- c(p_sem[[termo]], d)
  }
}
cat("\np_sem[['de']] (mostra d3 duplicado):\n")
print(p_sem[["de"]])

cat("\nCheckpoint 8: p_sem[['a']] (mostra d7 duplicado):\n")
print(p_sem[["a"]])

Postings de 'documentos':
[1] "d1" "d2" "d5" "d6"

Postings de 'modelo':
[1] "d2" "d3"

Postings de 'de' (stopword removida):
NULL

p_sem[['de']] (mostra d3 duplicado):
[1] "d1" "d2" "d3" "d3" "d6" "d8"

Checkpoint 8: p_sem[['a']] (mostra d7 duplicado):
[1] "d4" "d5" "d6" "d7" "d7"


## Módulo 9 — Buscando pelo índice
Execução de consultas multi-termos via `Reduce(intersect)` e verificação da cardinalidade de postagens.

In [10]:
# Módulo 9: Consultas eficientes via interseccao de postagens
busca_AND <- function(consulta) {
  termos <- prep(consulta)
  if (!all(termos %in% names(postings))) return(character(0))
  Reduce(intersect, postings[termos])
}

cat("busca_AND('modelo documentos'):\n")
print(busca_AND("modelo documentos"))

cat("\nbusca_AND('busca documentos'):\n")
print(busca_AND("busca documentos"))

cat("\nEstatisticas do indice:\n")
cat("Total de termos indexados:", length(postings), "(45 - 8 stopwords)\n")
cat("Termos com mais documentos (df):\n")
print(sort(lengths(postings), decreasing = TRUE)[1:4])

cat("\nCasos adicionais e Checkpoint 9:\n")
cat("busca_AND('Documentos')            : ", busca_AND("Documentos"), "\n")
cat("busca_AND('modelo xyz')             : ", busca_AND("modelo xyz"), "(vazio)\n")
cat("busca_AND('documentos modelo bm25') : ", busca_AND("documentos modelo bm25"), "(vazio)\n")
cat("busca_AND('Relevancia da busca')    : ", busca_AND("Relevancia da busca"), "\n")

# Exploracao: busca_OR com union
busca_OR <- function(consulta) {
  termos <- prep(consulta)
  termos_validos <- termos[termos %in% names(postings)]
  if (length(termos_validos) == 0) return(character(0))
  sort(unique(Reduce(union, postings[termos_validos])))
}
cat("\nExploracao - busca_OR('modelo busca'):\n")
print(busca_OR("modelo busca"))

busca_AND('modelo documentos'):
[1] "d2"

busca_AND('busca documentos'):
[1] "d5"

Estatisticas do indice:
Total de termos indexados: 37 (45 - 8 stopwords)
Termos com mais documentos (df):
documentos recuperacao  relevancia      modelo 
         4           2           2           2 

Casos adicionais e Checkpoint 9:
busca_AND('Documentos')            :  d1 d2 d5 d6 
busca_AND('modelo xyz')             :  (vazio)
busca_AND('documentos modelo bm25') :  (vazio)
busca_AND('Relevancia da busca')    :  d7 

Exploracao - busca_OR('modelo busca'):
[1] "d2" "d3" "d5" "d7"


## Fechamento da Sessão e Snapshot de Estado
Anexação da memória do R no consolidado.

In [11]:
# Salvamento de estado no consolidado
if (file.exists("aula03_consolidado.md")) {
  anexar_estado("aula03_consolidado.md")
  cat("Estado anexado a aula03_consolidado.md com sucesso!\n")
}
cat("Sessao teorica finalizada!\n")

Estado anexado a aula03_consolidado.md com sucesso!
Sessao teorica finalizada!
